# build_2khz_tensor_dict

Builds the `segfilt_2khz_emg_tensor_dict.pkl` file needed for the A10/A11/A12 Meta model ablations.

**Resuming from checkpoint:** The kernel crashed after saving `ppdsegraw1_2khz_EMG.json`.
Cells 2–3 check for saved checkpoints and skip the expensive work if found.
On a fresh run they execute the full pipeline automatically.

**Run order:** 0 → 1 → 2 → 3 → 4 (read output!) → 5 → 6 → 7

## 0. Imports

In [1]:
import json
import os
import pickle
import time
from pathlib import Path

import numpy as np
import torch
from scipy.signal import resample as scipy_resample

from shared_processing import (
    load_segraw_data,
    apply_filter_to_nested_dict,
    normalize_gestures_by_std_any_channels,
)

## 1. Config

In [2]:
RAW_DATA_DIR = r"C:\Users\kdmen\Box\Yamagami Lab\Data\2024_UIST_dataset\upload\segmented_raw_data"
SAVE_DIR     = r"C:\Users\kdmen\Box\Yamagami Lab\Data\Meta_Gesture_Project"

RAW_DICT_SAVE_PATH = SAVE_DIR + r"\segraw0_2khz_EMG.json"
PPD_DICT_SAVE_PATH = SAVE_DIR + r"\ppdsegraw1_2khz_EMG.json"
PKL_SAVE_PATH      = SAVE_DIR + r"\segfilt_2khz_emg_tensor_dict.pkl"

FS = 2000  # Hz — do not change

# Set from survey output (Cell 4). Previous run showed p10 = 4,340 → 4,300.
# Check the outlier report in Cell 4 before accepting this.
TARGET_LEN_SAMPLES = 4300

PIDS_IMPAIRED   = ['P102','P103','P104','P105','P106','P107','P108','P109','P110','P111',
                   'P112','P114','P115','P116','P118','P119','P121','P122','P123','P124',
                   'P125','P126','P127','P128','P131','P132']
PIDS_UNIMPAIRED = ['P004','P005','P006','P008','P010','P011']
PIDS_ALL        = PIDS_IMPAIRED + PIDS_UNIMPAIRED

GESTURE_TO_CLASS = {
    'close':         0,
    'delete':        1,
    'duplicate':     2,
    'move':          3,
    'open':          4,
    'pan':           5,
    'rotate':        6,
    'select-single': 7,
    'zoom-in':       8,
    'zoom-out':      9,
}

print(f"TARGET_LEN_SAMPLES : {TARGET_LEN_SAMPLES} samples = {TARGET_LEN_SAMPLES/FS*1000:.0f} ms")
print(f"PKL_SAVE_PATH      : {PKL_SAVE_PATH}")

TARGET_LEN_SAMPLES : 4300 samples = 2150 ms
PKL_SAVE_PATH      : C:\Users\kdmen\Box\Yamagami Lab\Data\Meta_Gesture_Project\segfilt_2khz_emg_tensor_dict.pkl


## 2. Load raw CSVs (or checkpoint)

Loads `segraw0_2khz_EMG.json` if it exists, otherwise loads from raw CSVs (~50 min).
Disable laptop sleep if running from raw CSVs.

In [3]:
if Path(RAW_DICT_SAVE_PATH).exists():
    print(f"Checkpoint found — loading {RAW_DICT_SAVE_PATH}")
    t0 = time.time()
    with open(RAW_DICT_SAVE_PATH, 'r') as f:
        nested_dict = json.load(f)
    print(f"Loaded in {time.time()-t0:.1f}s  |  {len(nested_dict)} participants")
else:
    print("No checkpoint — loading raw CSVs (~50 min). Disable laptop sleep now.")
    t0 = time.time()
    nested_dict = load_segraw_data(
        pIDs             = PIDS_ALL,
        data_dir_path    = RAW_DATA_DIR,
        modalities       = ["E"],
        expt_types       = ["experimenter-defined"],
        num_emg_channels = 16,
    )
    print(f"Done in {time.time()-t0:.1f}s")
    with open(RAW_DICT_SAVE_PATH, 'w') as f:
        json.dump(nested_dict, f)
    print(f"Saved → {RAW_DICT_SAVE_PATH}")

sample_pid     = list(nested_dict.keys())[0]
sample_gesture = list(nested_dict[sample_pid].keys())[0]
sample_trial   = list(nested_dict[sample_pid][sample_gesture].keys())[0]
print(f"Sample reference: {sample_pid} / {sample_gesture} / trial {sample_trial}")

Checkpoint found — loading C:\Users\kdmen\Box\Yamagami Lab\Data\Meta_Gesture_Project\segraw0_2khz_EMG.json
Loaded in 74.0s  |  32 participants
Sample reference: P102 / pan / trial 1


## 3. BPF + normalise (or checkpoint)

In [4]:
if Path(PPD_DICT_SAVE_PATH).exists():
    print(f"Checkpoint found — loading {PPD_DICT_SAVE_PATH}")
    t0 = time.time()
    with open(PPD_DICT_SAVE_PATH, 'r') as f:
        ppd_dict = json.load(f)
    print(f"Loaded in {time.time()-t0:.1f}s  |  {len(ppd_dict)} participants")
else:
    print("Applying BPF 20-450 Hz + mean subtraction...")
    t0 = time.time()
    filt_dict = apply_filter_to_nested_dict(
        nested_dict,
        normalization_method = "MEANSUBTRACTION",
        already_BPFd         = False,
    )
    print(f"  Done in {time.time()-t0:.1f}s")

    print("Per-gesture std normalisation...")
    t0 = time.time()
    ppd_dict = normalize_gestures_by_std_any_channels(filt_dict)
    print(f"  Done in {time.time()-t0:.1f}s")

    with open(PPD_DICT_SAVE_PATH, 'w') as f:
        json.dump(ppd_dict, f)
    print(f"Saved -> {PPD_DICT_SAVE_PATH}")

sample_data_ppd = ppd_dict[sample_pid][sample_gesture][sample_trial]["EMG"]
flat = [v for ch in sample_data_ppd for v in ch]
print(f"Spot-check std (should be ~1.0): {np.std(flat):.4f}")

Checkpoint found — loading C:\Users\kdmen\Box\Yamagami Lab\Data\Meta_Gesture_Project\ppdsegraw1_2khz_EMG.json
Loaded in 495.8s  |  32 participants
Spot-check std (should be ~1.0): 1.0000


## 4. Survey + outlier report

**Read this output before running Cell 5.** The previous run showed:
- Min = 1,580 samples (790 ms) and Max = 24,999 (12.5 s) — very wide range
- p10 = 4,340 → TARGET = 4,300

The outlier table below tells you *which* trials are short or extremely long.
If they look like data artifacts, consider excluding them before building the tensor_dict.
To exclude: add their `(pid, gesture, trial_key)` to an `EXCLUDE` set and skip them in Cell 5.

In [5]:
records = [
    (pid, gesture, trial, len(ppd_dict[pid][gesture][trial]["EMG"][0]))
    for pid in ppd_dict
    for gesture in ppd_dict[pid]
    for trial in ppd_dict[pid][gesture]
]
lengths = np.array([r[3] for r in records])

print("=== Trial Length Distribution ===")
for label, pct in [("Min",0),("p5",5),("p10",10),("p25",25),
                   ("Median",50),("p75",75),("p90",90),("p95",95),("Max",100)]:
    v = np.percentile(lengths, pct)
    print(f"  {label:<8}: {int(v):>7,}  ({v/FS*1000:.0f} ms)")
print(f"  Std     : {lengths.std():>7.0f}  ({lengths.std()/FS*1000:.0f} ms)")

median_len   = np.median(lengths)
short_cutoff = TARGET_LEN_SAMPLES
long_cutoff  = int(median_len * 3)

short_trials = [(p,g,t,l) for p,g,t,l in records if l < short_cutoff]
long_trials  = [(p,g,t,l) for p,g,t,l in records if l > long_cutoff]

print(f"\n=== Short trials (< TARGET={TARGET_LEN_SAMPLES}, will be upsampled) ===")
if short_trials:
    for p,g,t,l in sorted(short_trials, key=lambda x: x[3]):
        print(f"  {p:>5}  {g:<16}  trial {str(t):<4}  {l:>6,} samples  ({l/FS*1000:.0f} ms)")
    print(f"  Total: {len(short_trials)} trials")
else:
    print("  None.")

print(f"\n=== Long outliers (> 3x median = {long_cutoff:,}) ===")
if long_trials:
    for p,g,t,l in sorted(long_trials, key=lambda x: -x[3]):
        print(f"  {p:>5}  {g:<16}  trial {str(t):<4}  {l:>6,} samples  ({l/FS*1000:.0f} ms)")
    print(f"  Total: {len(long_trials)} trials")
else:
    print("  None.")

=== Trial Length Distribution ===
  Min     :   1,580  (790 ms)
  p5      :   3,620  (1810 ms)
  p10     :   4,340  (2170 ms)
  p25     :   5,079  (2540 ms)
  Median  :   6,519  (3260 ms)
  p75     :   7,960  (3980 ms)
  p90     :   9,879  (4940 ms)
  p95     :  11,079  (5540 ms)
  Max     :  24,999  (12500 ms)
  Std     :    2450  (1225 ms)

=== Short trials (< TARGET=4300, will be upsampled) ===
   P106  duplicate         trial 8      1,580 samples  (790 ms)
   P109  close             trial 7      1,580 samples  (790 ms)
   P109  close             trial 8      1,700 samples  (850 ms)
   P106  pan               trial 10     1,940 samples  (970 ms)
   P106  close             trial 5      2,060 samples  (1030 ms)
   P106  duplicate         trial 6      2,060 samples  (1030 ms)
   P106  select-single     trial 10     2,060 samples  (1030 ms)
   P109  open              trial 1      2,060 samples  (1030 ms)
   P106  open              trial 1      2,180 samples  (1090 ms)
   P106  open     

## 5. Resample and build tensor_dict

After reviewing the outlier report above, add any trials you want to exclude to `EXCLUDE` below.
Format: `{(pid, gesture_name, trial_key), ...}`  e.g. `{('P102', 'pan', '1')}`.
Leave empty to include everything.

In [6]:
# Add (pid, gesture_name, trial_key) tuples here to exclude specific trials.
# All values should be strings, matching the keys in ppd_dict.
# Example: EXCLUDE = {('P102', 'pan', '3'), ('P115', 'zoom-in', '7')}
EXCLUDE = set()


def resample_trial(emg_channels: list, target_len: int) -> np.ndarray:
    """
    Resample one trial to exactly target_len samples.
    Input : list of 16 lists, each length T_original  (channel-first)
    Output: np.ndarray (target_len, 16)               (channel-last)
    """
    arr = np.array(emg_channels, dtype=np.float32)     # (16, T_orig)
    if arr.shape[1] != target_len:
        arr = scipy_resample(arr, target_len, axis=1)  # (16, target_len)
    return arr.T.astype(np.float32)                    # (target_len, 16)


print(f"Building tensor_dict — TARGET_LEN_SAMPLES={TARGET_LEN_SAMPLES}")
if EXCLUDE:
    print(f"Excluding {len(EXCLUDE)} trials: {EXCLUDE}")
t0 = time.time()

tensor_dict = {}
pid_list = sorted(ppd_dict.keys())

for pid_idx, pid in enumerate(pid_list):
    tensor_dict[pid] = {}
    print(f"  {pid} ({pid_idx+1}/{len(pid_list)})...", end=" ", flush=True)

    for gesture_name, gesture_class in GESTURE_TO_CLASS.items():
        if gesture_name not in ppd_dict[pid]:
            print(f"\n    WARNING: '{gesture_name}' missing for {pid}, skipping.")
            continue

        trial_keys = sorted(ppd_dict[pid][gesture_name].keys())
        trial_keys = [k for k in trial_keys if (pid, gesture_name, k) not in EXCLUDE]

        trial_arrays = []
        for trial_key in trial_keys:
            emg_ch = ppd_dict[pid][gesture_name][trial_key]["EMG"]
            trial_arrays.append(resample_trial(emg_ch, TARGET_LEN_SAMPLES))

        emg_tensor = torch.from_numpy(np.stack(trial_arrays, axis=0)).float()

        assert emg_tensor.shape == (len(trial_keys), TARGET_LEN_SAMPLES, 16), (
            f"Shape error {pid}/{gesture_name}: {emg_tensor.shape}"
        )

        tensor_dict[pid][gesture_class] = {
            "emg":         emg_tensor,
            "imu":         None,
            "demo":        torch.zeros(12),
            "gest_ID":     gesture_class,
            "enc_gest_ID": gesture_class,
            "enc_pid":     pid_idx,
            "rep_indices": [int(k) if str(k).isdigit() else i + 1
                            for i, k in enumerate(trial_keys)],
        }

    print("done")

print(f"\nFinished in {time.time()-t0:.1f}s")

Building tensor_dict — TARGET_LEN_SAMPLES=4300
  P004 (1/32)... done
  P005 (2/32)... done
  P006 (3/32)... done
  P008 (4/32)... done
  P010 (5/32)... done
  P011 (6/32)... done
  P102 (7/32)... done
  P103 (8/32)... done
  P104 (9/32)... done
  P105 (10/32)... done
  P106 (11/32)... done
  P107 (12/32)... done
  P108 (13/32)... done
  P109 (14/32)... done
  P110 (15/32)... done
  P111 (16/32)... done
  P112 (17/32)... done
  P114 (18/32)... done
  P115 (19/32)... done
  P116 (20/32)... done
  P118 (21/32)... done
  P119 (22/32)... done
  P121 (23/32)... done
  P122 (24/32)... done
  P123 (25/32)... done
  P124 (26/32)... done
  P125 (27/32)... done
  P126 (28/32)... done
  P127 (29/32)... done
  P128 (30/32)... done
  P131 (31/32)... done
  P132 (32/32)... done

Finished in 37.3s


## 6. Verify

In [7]:
print("=== Verification ===")
pids = list(tensor_dict.keys())
print(f"Participants : {len(pids)}")
print(f"Classes for {pids[0]}: {sorted(tensor_dict[pids[0]].keys())}")

errors = []
for pid in pids:
    for cls in tensor_dict[pid]:
        emg = tensor_dict[pid][cls]["emg"]
        if emg.shape[1] != TARGET_LEN_SAMPLES or emg.shape[2] != 16:
            errors.append(f"{pid}/class{cls}: shape {emg.shape}")
        if torch.isnan(emg).any() or torch.isinf(emg).any():
            errors.append(f"{pid}/class{cls}: NaN or Inf")

if errors:
    for e in errors:
        print(f"  ERROR: {e}")
else:
    print("All shape and NaN/Inf checks passed.")

sample_cls = sorted(tensor_dict[pids[0]].keys())[0]
sample_emg = tensor_dict[pids[0]][sample_cls]["emg"]
per_trial_stds = sample_emg.std(dim=[1, 2])
print(f"\nSample {pids[0]}/class{sample_cls} shape: {sample_emg.shape}")
print(f"Per-trial std: mean={per_trial_stds.mean():.3f}, "
      f"min={per_trial_stds.min():.3f}, max={per_trial_stds.max():.3f}")
print("(should be ~1.0)")

=== Verification ===
Participants : 32
Classes for P004: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
All shape and NaN/Inf checks passed.

Sample P004/class0 shape: torch.Size([10, 4300, 16])
Per-trial std: mean=1.000, min=1.000, max=1.000
(should be ~1.0)


## 7. Save

In [8]:
print(f"Saving -> {PKL_SAVE_PATH}")
t0 = time.time()

with open(PKL_SAVE_PATH, "wb") as f:
    pickle.dump({"data": tensor_dict}, f, protocol=4)

size_mb = os.path.getsize(PKL_SAVE_PATH) / 1e6
print(f"Done in {time.time()-t0:.1f}s  |  File size: {size_mb:.0f} MB")
print()
print("Next steps:")
print(f"  1. scp the file to your cluster:")
print(f"       scp '{PKL_SAVE_PATH}'")
print(f"           km82@<cluster>:/rhf/allocations/my13/div-emg/dataset/")
print()
print(f"  2. In A10_A11_A12_meta_pretrained.py set:")
print(f"       EMG_2KHZ_PKL_PATH = Path('/rhf/allocations/my13/div-emg/dataset/segfilt_2khz_emg_tensor_dict.pkl')")
print(f"       EMG_2KHZ_SEQ_LEN  = {TARGET_LEN_SAMPLES}")

Saving -> C:\Users\kdmen\Box\Yamagami Lab\Data\Meta_Gesture_Project\segfilt_2khz_emg_tensor_dict.pkl
Done in 2.3s  |  File size: 881 MB

Next steps:
  1. scp the file to your cluster:
       scp 'C:\Users\kdmen\Box\Yamagami Lab\Data\Meta_Gesture_Project\segfilt_2khz_emg_tensor_dict.pkl'
           km82@<cluster>:/rhf/allocations/my13/div-emg/dataset/

  2. In A10_A11_A12_meta_pretrained.py set:
       EMG_2KHZ_PKL_PATH = Path('/rhf/allocations/my13/div-emg/dataset/segfilt_2khz_emg_tensor_dict.pkl')
       EMG_2KHZ_SEQ_LEN  = 4300
